# Verdant Training Harness
Minimal live interaction harness for Colab.


In [ ]:
from verdant.system import VerdantSystem
from verdant.io import HumanTextInputAdapter, ConversationalOutputAdapter, QueryInterface, InteractionService

CHECKPOINT_PATH = None  # set to a checkpoint path string to restore
system = VerdantSystem()
if CHECKPOINT_PATH:
    system.load_state(CHECKPOINT_PATH)

input_adapter = HumanTextInputAdapter(person_id='will', session_id='colab_session')
output_adapter = ConversationalOutputAdapter()
query = QueryInterface(system)
system.register_adapter(input_adapter)
system.register_output_adapter(output_adapter)
service = InteractionService(system, input_adapter, output_adapter, query, base_dir='interactions')
print('Harness ready')


In [ ]:
text = 'I want to build trust over time'
response = service.turn('will', text)
print('Response:', response)
print(query.query(text))


In [ ]:
from pathlib import Path
file_path = Path('/content/input.txt')  # upload a .txt in Colab
before_nodes = set(system.memory_web.list_concepts())
if file_path.exists():
    for line in file_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line:
            service.turn('will', line)
after_nodes = set(system.memory_web.list_concepts())
print('New concepts learned:', len(after_nodes - before_nodes))


In [ ]:
concept = 'trust'
print(query.inspect_concept(concept))


In [ ]:
metrics = system.get_metrics()
basin_map = {}
for b in metrics.get('basins', []):
    bid = str(b.get('basin_id'))
    for n in b.get('nodes', []):
        basin_map[n] = bid
recent = query.recent_activations(20)
for item in recent:
    c = item['concept']
    print(c, item['activation'], basin_map.get(c, 'none'))
